# 02 · Inspect YOLO-OBB tiles

Draws the labels written by `trogs prepare yolo` back onto random tiles, to check that boxes
were clipped and normalised correctly. The red dot marks each box's first corner (orientation).

In [ ]:
import os
from pathlib import Path

# Run from the repository root so the default config paths resolve.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("Working directory:", Path.cwd())

In [ ]:
import glob
import os
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np

DATASET_DIR = "data/processed/greek_letters_obb"
SPLIT = "val"  # "train" or "val"
NUM_SAMPLES = 3

In [ ]:
image_paths = glob.glob(os.path.join(DATASET_DIR, "images", SPLIT, "*.jpg"))
samples = random.sample(image_paths, min(len(image_paths), NUM_SAMPLES))

plt.figure(figsize=(15, 5 * len(samples)))
for i, img_path in enumerate(samples):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    label_path = os.path.join(DATASET_DIR, "labels", SPLIT, os.path.basename(img_path).replace(".jpg", ".txt"))
    with open(label_path) as f:
        for line in f:
            coords = np.array([float(p) for p in line.split()[1:]]).reshape(-1, 2)
            points = (coords * [w, h]).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(img, [points], isClosed=True, color=(0, 255, 0), thickness=2)
            cv2.circle(img, tuple(points[0][0]), 5, (255, 0, 0), -1)
    plt.subplot(len(samples), 1, i + 1)
    plt.imshow(img)
    plt.title(os.path.basename(img_path))
    plt.axis("off")
plt.tight_layout()
plt.show()